# 501 — Mock Login Smoke Tests

Phase-1 mock authentication — direct validation of auth helpers and session-state logic.

**Scope**
- Mock user definitions
- `authenticate()` — success and failure paths
- `AuthenticatedUser` object shape
- `state` auth helpers (login / logout / error)
- Dev bypass helpers
- Verify auth state does NOT live in `AppComponents` (cached resource)

**Note:** Full Streamlit UI automation is not practical in notebook form.
Cells marked **[MANUAL UI CHECK]** describe what to verify by running the app.
All other cells execute automatically and raise `AssertionError` on failure.

In [1]:
# Bootstrap — make 'src' importable from the notebooks/ directory
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Repo root:", repo_root)

Repo root: /Users/emilpastor/Documents/github/entity-risk-ai


## Test 1 — Mock user definitions are loadable

In [2]:
from src.app.auth import get_mock_users

users = get_mock_users()

assert "jr_analyst" in users, "jr_analyst not found in mock users"
assert "sr_analyst" in users, "sr_analyst not found in mock users"

assert users["jr_analyst"]["role"] == "jr_risk_analyst"
assert users["sr_analyst"]["role"] == "sr_risk_analyst"

assert "password" in users["jr_analyst"]
assert "password" in users["sr_analyst"]

print("PASS — mock user definitions exist and have expected shape")
for username, entry in users.items():
    print(f"  {username}: role={entry['role']}")

PASS — mock user definitions exist and have expected shape
  jr_analyst: role=jr_risk_analyst
  sr_analyst: role=sr_risk_analyst


## Test 2 — Successful login for both demo users

In [3]:
from src.app.auth import authenticate, AuthenticatedUser

# jr_analyst
jr = authenticate("jr_analyst", "demo123")
assert jr is not None, "jr_analyst login returned None"
assert isinstance(jr, AuthenticatedUser)
assert jr.user_id == "jr_analyst"
assert jr.role == "jr_risk_analyst"
print(f"PASS — jr_analyst authenticated: role={jr.role}, provider={jr.auth_provider}")

# sr_analyst
sr = authenticate("sr_analyst", "demo123")
assert sr is not None, "sr_analyst login returned None"
assert isinstance(sr, AuthenticatedUser)
assert sr.user_id == "sr_analyst"
assert sr.role == "sr_risk_analyst"
print(f"PASS — sr_analyst authenticated: role={sr.role}, provider={sr.auth_provider}")

PASS — jr_analyst authenticated: role=jr_risk_analyst, provider=mock
PASS — sr_analyst authenticated: role=sr_risk_analyst, provider=mock


## Test 3 — Failed login for invalid credentials

In [4]:
from src.app.auth import authenticate

cases = [
    ("jr_analyst",   "wrong_password"),
    ("nonexistent",  "demo123"),
    ("",             ""),
    ("SR_ANALYST",   "demo123"),   # case-sensitive
    ("jr_analyst",   ""),
]

for username, password in cases:
    result = authenticate(username, password)
    assert result is None, f"Expected None for ({username!r}, {password!r}), got {result}"
    print(f"PASS — ({username!r}, {password!r}) correctly rejected")

PASS — ('jr_analyst', 'wrong_password') correctly rejected
PASS — ('nonexistent', 'demo123') correctly rejected
PASS — ('', '') correctly rejected
PASS — ('SR_ANALYST', 'demo123') correctly rejected
PASS — ('jr_analyst', '') correctly rejected


## Test 4 — AuthenticatedUser object shape

In [5]:
import uuid
from src.app.auth import authenticate

user = authenticate("jr_analyst", "demo123")
assert user is not None

# Required fields
assert hasattr(user, "user_id"),       "missing user_id"
assert hasattr(user, "role"),          "missing role"
assert hasattr(user, "auth_provider"), "missing auth_provider"
assert hasattr(user, "session_id"),    "missing session_id"
assert hasattr(user, "metadata"),      "missing metadata"

# Values
assert isinstance(user.user_id, str) and user.user_id != ""
assert isinstance(user.role, str) and user.role != ""
assert user.auth_provider == "mock"
assert isinstance(user.session_id, str)
uuid.UUID(user.session_id)            # must be valid UUID
assert isinstance(user.metadata, dict)

# Each call generates a unique session_id
user2 = authenticate("jr_analyst", "demo123")
assert user.session_id != user2.session_id, "session_id must be unique per authentication"

print("PASS — AuthenticatedUser has all required fields with correct types")
print(f"  user_id={user.user_id!r}, role={user.role!r}, auth_provider={user.auth_provider!r}")
print(f"  session_id={user.session_id!r}")

PASS — AuthenticatedUser has all required fields with correct types
  user_id='jr_analyst', role='jr_risk_analyst', auth_provider='mock'
  session_id='67fd5f72-1f80-4ecd-88e7-5ade14297b46'


## Test 5 — state auth helpers: login / logout / error

The `state` module writes to `st.session_state`, which requires a running
Streamlit context.  We test the helpers here by patching `st.session_state`
with a plain dict (the helpers use only dict-like access).

In [6]:
import types
import importlib

# Patch st.session_state with a plain dict so state helpers run outside Streamlit.
import streamlit as st
_real_session_state = st.session_state

fake_ss = {}
st.session_state = fake_ss  # type: ignore[assignment]

try:
    import src.app.state as state
    importlib.reload(state)          # reload so module-level refs pick up fake_ss

    # Seed defaults
    state.init()
    assert not state.is_authenticated(), "should start unauthenticated"
    assert state.get_authenticated_user() is None
    assert state.get_auth_error() is None
    print("PASS — initial state is unauthenticated")

    # login
    from src.app.auth import authenticate
    user = authenticate("sr_analyst", "demo123")
    state.login(user)
    assert state.is_authenticated()
    assert state.get_authenticated_user() is user
    assert state.get_auth_error() is None
    print("PASS — login() sets authenticated=True and stores user")

    # auth error
    state.set_auth_error("Bad credentials")
    assert state.get_auth_error() == "Bad credentials"
    state.set_auth_error(None)
    assert state.get_auth_error() is None
    print("PASS — set_auth_error / get_auth_error round-trip")

    # logout
    state.logout()
    assert not state.is_authenticated()
    assert state.get_authenticated_user() is None
    assert fake_ss.get("auth_dev_bypass") == False
    print("PASS — logout() clears all auth state")

finally:
    st.session_state = _real_session_state  # type: ignore[assignment]
    importlib.reload(state)                 # restore real module

PASS — initial state is unauthenticated
PASS — login() sets authenticated=True and stores user
PASS — set_auth_error / get_auth_error round-trip
PASS — logout() clears all auth state


## Test 6 — Dev bypass helpers

In [7]:
import os
from src.app.auth import is_dev_bypass_enabled, dev_bypass_user

# Default: bypass off
os.environ.pop("DEV_BYPASS_AUTH", None)
assert not is_dev_bypass_enabled(), "bypass should be off when env var is unset"
print("PASS — bypass disabled when DEV_BYPASS_AUTH is unset")

# Enable via true
for val in ("true", "True", "TRUE", "1", "yes"):
    os.environ["DEV_BYPASS_AUTH"] = val
    assert is_dev_bypass_enabled(), f"bypass should be on for DEV_BYPASS_AUTH={val!r}"
print("PASS — bypass enabled for true/1/yes variants")

# Disable
for val in ("false", "0", "", "no"):
    os.environ["DEV_BYPASS_AUTH"] = val
    assert not is_dev_bypass_enabled(), f"bypass should be off for DEV_BYPASS_AUTH={val!r}"
print("PASS — bypass disabled for false/0/no variants")

# dev_bypass_user shape
os.environ.pop("DEV_BYPASS_AUTH", None)
bypass = dev_bypass_user()
assert bypass.user_id == "dev_bypass"
assert bypass.role == "sr_risk_analyst"
assert bypass.auth_provider == "dev_bypass"
assert bypass.metadata.get("bypass") is True
print(f"PASS — dev_bypass_user(): user_id={bypass.user_id!r}, role={bypass.role!r}")

PASS — bypass disabled when DEV_BYPASS_AUTH is unset
PASS — bypass enabled for true/1/yes variants
PASS — bypass disabled for false/0/no variants
PASS — dev_bypass_user(): user_id='dev_bypass', role='sr_risk_analyst'


## Test 7 — Auth state is NOT stored in cached AppComponents

This is a structural check: `AppComponents` must not contain any auth-related
fields.  Auth state belongs exclusively in `st.session_state`.

In [8]:
import dataclasses
from src.app.factory import AppComponents

field_names = {f.name for f in dataclasses.fields(AppComponents)}
print("AppComponents fields:", sorted(field_names))

auth_keywords = {"auth", "user", "login", "session", "role", "authenticated"}
collisions = {
    name for name in field_names
    if any(kw in name.lower() for kw in auth_keywords)
}

assert not collisions, (
    f"AppComponents must not carry auth state, found: {collisions}. "
    "Auth belongs in st.session_state only."
)
print("PASS — AppComponents contains no auth-related fields")

AppComponents fields: ['ai_client', 'graph_agent', 'mcp_client', 'orchestrator', 'planner', 'repo', 'risk_agent', 'trace_agent', 'trace_repo', 'trace_service']
PASS — AppComponents contains no auth-related fields


## Manual UI smoke checks

Run `streamlit run app.py` and verify the following manually.

| # | Scenario | Expected |
|---|----------|----------|
| 1 | Open app fresh | Login gate shown; main app not visible |
| 2 | Submit empty form | Error: "Invalid username or password." |
| 3 | Submit wrong password | Error: "Invalid username or password." |
| 4 | Login as `jr_analyst / demo123` | Main app renders; sidebar shows `jr_analyst`, role `jr_risk_analyst` |
| 5 | Login as `sr_analyst / demo123` | Main app renders; sidebar shows `sr_analyst`, role `sr_risk_analyst` |
| 6 | Click "Sign out" | Returns to login gate; main app gone |
| 7 | Set `DEV_BYPASS_AUTH=true` and restart | "Enter as dev user" button appears below the form |
| 8 | Click dev bypass | Enters app as `dev_bypass` / `sr_risk_analyst` |
| 9 | Run an investigation after login | Orchestrator, agents, progress, graph all work normally |

## Summary

All automated cells above validate:

- Mock user registry is importable and well-formed
- `authenticate()` succeeds for valid credentials and returns `None` for all invalid variants
- `AuthenticatedUser` carries `user_id`, `role`, `auth_provider`, `session_id`, `metadata`
- `session_id` is a unique UUID generated per authentication call
- `state.login()` / `state.logout()` / `state.set_auth_error()` behave correctly against a simulated session state dict
- Dev bypass is gated behind the `DEV_BYPASS_AUTH` env var; `dev_bypass_user()` returns expected shape
- `AppComponents` (the `@st.cache_resource` object) contains zero auth-related fields

Manual UI smoke checks are listed in the cell above.